This is the group project submission for Merina, Gabriel, and Dominic.

We chose to work on the Snowflake Fingerprintability project on the nPrint PCAPML leaderboard. It was aimed to differentiate Snowflake Tor DTLS handshakes from other WebRTS traffic. DTLS (Datagram Transport Layer Security) is one of the main forms of security protocols used by WebRTC traffic. There are 7 different types of traffic analyzed in the dataset we are using: Snowflake, Facebook Messenger, Discord, and Google hangouts, across Google Chrome and Firefox browsers. According to the paper cited in the study.  

There are 2 main, important differences between the Snowflake traffic and the other types of connections. The first, the Snowflake handshakes contain significantly more packets: 13.2 to around 5 for the other types of traffic(Macmillan et al.). The other is that there are 2 features that are present in Snowflake traffic that is missing from all the other traffic: "Server Message Sequence: '1'" and "supported_groups." There is also a feature, "renegotiation info," that is missing from Snowflake traffic, but included in the other types of traffic. A third, now-spurious correlation is that all Snowflake traffic occured on Firefox: The extension is now available on Chrome as well, so this is no longer a viable way to differentiate this traffic.  

To differentiate the traffic, we will:

- Label all of the packet capture data from the given handshake sets, so we can create training data from it
- Train the Model on the labeled handshake data from the paper's github repo
- Load the packet capture of the case study as test data
- Test the model on the data
- Visualize the results (Accuracy, AUC Graph, Confusion Matrix)

In [14]:
import glob
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from netml.pparser.parser import PCAP
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score, classification_report

In [15]:
DATA_DIR = "webrtc-handshakes"
CATEGORIES = ["discord", "facebook", "google", "snowflake"]
FEAT_TYPE = "STATS"
FEATURE_COLS = [
    "duration", "pkts_rate", "bytes_rate", "size_mean", "size_std",
    "size_q1", "size_q2", "size_q3", "size_min", "size_max",
    "num_pkts", "num_bytes",
]
#Initializing constants for the model to extract useful feautres
# and be able to classify based on the categories avaialable

In [16]:
def pcap_to_flow_features(pcap_file, feat_type=FEAT_TYPE, flow_ptks_thres=2):
    """Parse one pcap file into netml flows and return (features, fids)."""
    pp = PCAP(pcap_file, flow_ptks_thres=flow_ptks_thres, verbose=0)
    pp.pcap2flows()
    if len(pp.flows) == 0:
        return np.empty((0, len(FEATURE_COLS))), []
    pp.flow2features(feat_type=feat_type)
    return pp.features, pp.fids

In [17]:
"""
This block will take all the pcap files in the directory, extract the flows,
and extract features from the flows. It will create a dataframe from a list of
lists containing specific features and the labels of the data. This will help us 
create training data for our model to classify the data based on the features 
extracted from the flows.
"""

records = []
for category in CATEGORIES:
    pcap_files = glob.glob(os.path.join(DATA_DIR, category, "*.pcap"))
    for pcap_file in pcap_files:
        features, fids = pcap_to_flow_features(pcap_file)
        for feat, fid in zip(features, fids):
            src_ip, dst_ip, src_port, dst_port, protocol = fid
            records.append({
                "file": pcap_file,
                "label": category,
                "src_ip": src_ip,
                "dst_ip": dst_ip,
                "src_port": src_port,
                "dst_port": dst_port,
                "protocol": protocol,
                **dict(zip(FEATURE_COLS, feat)),
            })

flows_df = pd.DataFrame.from_records(records)
print(flows_df.shape)
print(flows_df["label"].value_counts())

(8773, 19)
label
discord      3346
snowflake    1932
facebook     1824
google       1671
Name: count, dtype: int64


Our data processing relies on the netml STATs features, which is distinct from the original research paper's extraction from DTLS handshakes. Considering our group's skill level and the timeframe of the project, we have pivoted our project objective from replication the research paper's results, to comparing of a simpler netml approach with a different model could produce a comparable macro f1 score.

We also have opted for a traditional train-test split for training the model because in the **Evaluating Snowflake as an Indistinguishable
Censorship Circumvention Tool** research paper that seemed to be the methodology when they classified handshakes. It was unclear why there  was also Snowflake case study data included in their repository; however, based on training and testing with only the handshakes versus the handshakes and the case study data, we can conclude that the findings in this research paper were contingent on using only the handshakes.

In [18]:
#Declare training set from the flows_df
X_train = flows_df.loc[:, flows_df.columns != "label"]
y_train = flows_df["label"]

In [ ]:
#Need to encode the categorical values, one hot for replicating the research procedure
onc = OneHotEncoder(handle_unknown="infrequent_if_exist")

cat_cols = ["src_ip", "dst_ip"]

#Used label encoder for y, so that could keep data one-dimensional
le = LabelEncoder()

In [20]:
#Dropping columns and encoding training set
#X_train = X_train.drop(columns = "file")

#When encoding, use fit_transform
#X_train = X_train.toarray()
X_train = onc.fit_transform(X_train[cat_cols])

y_train = le.fit_transform(y_train)

In [21]:
#Initializing XGBoost Classifier and fitting it to training set
clf = GradientBoostingClassifier(n_estimators=200, learning_rate=0.2, max_depth=3, random_state=0)
clf.fit(X_train, y_train)

GradientBoostingClassifier(learning_rate=0.2, n_estimators=200, random_state=0)

We selected XGBoost Classifier because it is also an ensemble method, like Random Forest from the research paper, and it provides a comparison between boosting and bagging ensemble methods. XGBoost and Random Forest are both designed to prevent overfitting and utilize multiple decision trees to arrive to their classification.

In [ ]:
#Importing testing set, turning it to flows
pcap = PCAP("app_case_study.pcapng", flow_ptks_thres=2, verbose=0)
pcap.pcap2flows()

In [23]:
#Getting the features from the flows
pcap.flow2features(feat_type="STATS")
pcap.features

array([[2.34375000e-02, 8.53333333e+01, 5.81973333e+04, ...,
        1.16100000e+03, 2.00000000e+00, 1.36400000e+03],
       [2.34375000e-02, 8.53333333e+01, 7.09973333e+04, ...,
        1.00400000e+03, 2.00000000e+00, 1.66400000e+03],
       [2.73437500e-02, 7.31428571e+01, 4.98834286e+04, ...,
        1.16100000e+03, 2.00000000e+00, 1.36400000e+03],
       ...,
       [4.09179688e-01, 1.95513126e+01, 6.64255847e+03, ...,
        6.36000000e+02, 8.00000000e+00, 2.71800000e+03],
       [4.27734375e-01, 1.63652968e+01, 3.41333333e+03, ...,
        7.85000000e+02, 7.00000000e+00, 1.46000000e+03],
       [1.26953125e-02, 2.36307692e+02, 5.45870769e+04, ...,
        2.31000000e+02, 3.00000000e+00, 6.93000000e+02]])

In [24]:
#Adapting the training set function for converting pcaps into dfs, but no loop
records = []

for feat, fid in zip(pcap.features, pcap.fids):
    src_ip, dst_ip, src_port, dst_port, protocol = fid
    records.append({
        "label": "snowflake",
        "src_ip": src_ip,
        "dst_ip": dst_ip,
        "src_port": src_port,
        "dst_port": dst_port,
        "protocol": protocol,
        **dict(zip(FEATURE_COLS, feat))
    })
test_df = pd.DataFrame.from_records(records)

In [25]:
#Checking output of testing set
test_df

,label,src_ip,dst_ip,src_port,dst_port,protocol,duration,pkts_rate,bytes_rate,size_mean,size_std,size_q1,size_q2,size_q3,size_min,size_max,num_pkts,num_bytes
0,snowflake,74.125.250.71,192.168.7.222,19305,55937,17,0.023438,85.333333,58197.333333,682.000000,479.000000,442.50,682.0,921.50,203.0,1161.0,2.0,1364.0
1,snowflake,192.168.7.222,74.125.250.71,55937,19305,17,0.023438,85.333333,70997.333333,832.000000,172.000000,746.00,832.0,918.00,660.0,1004.0,2.0,1664.0
2,snowflake,74.125.250.26,192.168.7.222,19305,54537,17,0.027344,73.142857,49883.428571,682.000000,479.000000,442.50,682.0,921.50,203.0,1161.0,2.0,1364.0
3,snowflake,192.168.7.222,74.125.250.26,54537,19305,17,0.026367,75.851852,63070.814815,831.500000,172.500000,745.25,831.5,917.75,659.0,1004.0,2.0,1663.0
4,snowflake,74.125.250.71,192.168.7.222,19305,54510,17,0.023438,85.333333,58538.666667,686.000000,483.000000,444.50,686.0,927.50,203.0,1169.0,2.0,1372.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11008,snowflake,47.184.20.184,192.168.1.26,56208,59420,17,0.198242,30.266010,11279.133005,372.666667,186.383714,236.00,251.0,539.75,231.0,636.0,6.0,2236.0
11009,snowflake,192.168.1.26,47.184.20.184,59420,56208,17,0.199219,35.137255,7323.607843,208.428571,235.160649,117.00,117.0,117.00,90.0,784.0,7.0,1459.0
11010,snowflake,176.163.27.124,192.168.1.17,57327,57807,17,0.409180,19.551313,6642.558473,339.750000,171.259124,231.00,251.0,347.25,231.0,636.0,8.0,2718.0
11011,snowflake,192.168.1.17,176.163.27.124,57807,57327,17,0.427734,16.365297,3413.333333,208.571429,235.510301,117.00,117.0,117.00,90.0,785.0,7.0,1460.0


In [26]:
#Initializing testing set and encoding
#Use transform for encoding testing set

X_test = test_df.loc[:, test_df.columns != "label"]
X_test = onc.transform(X_test[cat_cols])

y_test = test_df["label"]
y_test = le.transform(y_test)

In [27]:
#Getting predicted values
y_pred = clf.predict(X_test)

In [28]:
#Randomized search cv, written with assistance of Claude Sonnet 5 for ranges
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "max_depth": [2, 3, 4, 5, 6],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "subsample": [0.6, 0.8, 1.0],
    "max_features": ["sqrt", "log2", None]
}

search = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(random_state=0),
    param_distributions=param_dist,
    n_iter=40,
    scoring="f1_macro",
    cv=5,
    random_state=0,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)

best_clf = search.best_estimator_

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best params: {'subsample': 1.0, 'n_estimators': 500, 'min_samples_split': 20, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 5, 'learning_rate': 0.2}


We re-trained the model with the paramters from the cross-validation process, so that we could maximize the average macro-weighted f1 score, just like the research paper. We selected RandomizedSearchCV because it is a scalable k-fold cross validation method, which tests parameters across their provided distribution ranges. We specifically used 5-fold, so that it was consistent with the research paper evaluation.

In [29]:
#Initializing XGBoost Classifier with best parameters
best_clf = GradientBoostingClassifier(subsample = 1.0, 
                    n_estimators = 500, 
                    min_samples_split = 20, 
                    min_samples_leaf = 1, 
                    max_features = None, 
                    max_depth = 5, 
                    learning_rate = 0.2)

In [30]:
#Fitting best XGBoost Classifier
best_clf.fit(X_train, y_train)

GradientBoostingClassifier(learning_rate=0.2, max_depth=5, min_samples_split=20,
                           n_estimators=500)

In [31]:
#Getting predicted values from best XGBoost Classifier
y_pred = best_clf.predict(X_test)

@article{macmillan2020evaluating,
title={Evaluating snowflake as an indistinguishable censorship circumvention tool},
author={MacMillan, Kyle and Holland, Jordan and Mittal, Prateek},
journal={arXiv preprint arXiv:2008.03254},
year={2020}
}